# Sentiment Analysis with RNNs — Women's Clothing E-Commerce Reviews

**Goal**: Predict whether a customer **recommends** a product (`Recommended IND`: 0 or 1) from their review text.

### Notebook Outline
1. Data Loading & Exploratory Data Analysis
2. Text Preprocessing & Vocabulary Building
3. PyTorch Dataset & DataLoader
4. Model Definitions (Vanilla RNN, LSTM, GRU)
5. Training Loop with Validation
6. Evaluation & Model Comparison
7. Inference on New Reviews

In [ ]:
import os
import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

---
## 1. Data Loading & EDA

In [ ]:
DATA_PATH = os.path.join('Dataset', 'Womens Clothing E-Commerce Reviews.csv')
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print(f"\nTarget distribution (Recommended IND):")
print(df['Recommended IND'].value_counts())
print(f"\nRecommendation rate: {df['Recommended IND'].mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target distribution
df['Recommended IND'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Recommendation Distribution')
axes[0].set_xticklabels(['Not Recommended', 'Recommended'], rotation=0)

# Rating distribution
df['Rating'].value_counts().sort_index().plot.bar(ax=axes[1], color='#3498db')
axes[1].set_title('Rating Distribution')
axes[1].set_xlabel('Rating')

# Review length distribution
df['review_len'] = df['Review Text'].dropna().str.split().str.len()
df['review_len'].plot.hist(bins=50, ax=axes[2], color='#9b59b6', edgecolor='white')
axes[2].set_title('Review Length (words)')
axes[2].axvline(df['review_len'].median(), color='red', linestyle='--', label=f'Median: {df["review_len"].median():.0f}')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Rating vs Recommendation
ct = pd.crosstab(df['Rating'], df['Recommended IND'], normalize='index')
ct.plot.bar(stacked=True, figsize=(8, 4), color=['#e74c3c', '#2ecc71'])
plt.title('Recommendation Rate by Rating')
plt.ylabel('Proportion')
plt.legend(['Not Recommended', 'Recommended'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = df.dropna(subset=['Review Text'])
df['clean_text'] = df['Review Text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0]

print(f"Cleaned dataset shape: {df.shape}")
print(f"\nSample review (original):")
print(df['Review Text'].iloc[0][:200])
print(f"\nSample review (cleaned):")
print(df['clean_text'].iloc[0][:200])

In [ ]:
# Train / Validation / Test split
texts = df['clean_text'].tolist()
labels = df['Recommended IND'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")
print(f"Train positive rate: {np.mean(y_train):.2%}")

In [ ]:
class Vocabulary:
    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'

    def __init__(self, max_size=25_000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.token2idx = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2token = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}

    def build(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        idx = len(self.token2idx)
        for word, freq in counter.most_common(self.max_size):
            if freq < self.min_freq:
                continue
            if word not in self.token2idx:
                self.token2idx[word] = idx
                self.idx2token[idx] = word
                idx += 1

    def encode(self, text, max_len):
        tokens = text.split()[:max_len]
        indices = [self.token2idx.get(t, 1) for t in tokens]
        return indices + [0] * (max_len - len(indices))

    def __len__(self):
        return len(self.token2idx)

vocab = Vocabulary(max_size=25_000, min_freq=2)
vocab.build(X_train)
print(f"Vocabulary size: {len(vocab):,}")

---
## 3. PyTorch Dataset & DataLoader

In [ ]:
MAX_LEN = 200
BATCH_SIZE = 64

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.vocab.encode(self.texts[idx], self.max_len)
        return (
            torch.tensor(enc, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float),
        )

train_ds = ReviewDataset(X_train, y_train, vocab)
val_ds   = ReviewDataset(X_val, y_val, vocab)
test_ds  = ReviewDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# Verify
sample_text, sample_label = next(iter(train_loader))
print(f"Batch text shape : {sample_text.shape}")
print(f"Batch label shape: {sample_label.shape}")

---
## 4. Model Definitions

In [ ]:
EMBED_DIM  = 128
HIDDEN_DIM = 128
N_LAYERS   = 2
DROPOUT    = 0.3


class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.rnn(embedded)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(hidden)).squeeze(1)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(hidden)).squeeze(1)


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(hidden)).squeeze(1)

---
## 5. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        preds = model(texts)
        loss = criterion(preds, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        predicted = (torch.sigmoid(preds) >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += len(labels)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    for texts, labels in loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        logits = model(texts)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        probs = torch.sigmoid(logits)
        predicted = (probs >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += len(labels)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_probs), np.array(all_labels)

In [ ]:
EPOCHS = 5
LR = 1e-3

models_config = {
    'Vanilla RNN': VanillaRNN,
    'LSTM': LSTMClassifier,
    'GRU': GRUClassifier,
}

history = {}  # store training history for each model
best_models = {}

for name, ModelClass in models_config.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"{'='*60}")

    model = ModelClass(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    train_losses, val_losses, val_accs = [], [], []
    best_val_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_models[name] = model.state_dict().copy()

        print(f"  Epoch {epoch}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"{elapsed:.1f}s")

    history[name] = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accs}

---
## 6. Evaluation & Model Comparison

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, h in history.items():
    axes[0].plot(h['train_loss'], label=name, linewidth=2)
    axes[1].plot(h['val_loss'], label=name, linewidth=2)
    axes[2].plot(h['val_acc'], label=name, linewidth=2)

for ax, title in zip(axes, ['Training Loss', 'Validation Loss', 'Validation Accuracy']):
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Test evaluation for all models
criterion = nn.BCEWithLogitsLoss()
test_results = {}

for name, ModelClass in models_config.items():
    model = ModelClass(
        vocab_size=len(vocab), embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, dropout=DROPOUT
    ).to(DEVICE)
    model.load_state_dict(best_models[name])

    test_loss, test_acc, probs, labels = evaluate(model, test_loader, criterion)
    auc = roc_auc_score(labels, probs)
    test_results[name] = {'acc': test_acc, 'auc': auc, 'probs': probs, 'labels': labels}

    print(f"\n{name}:  Test Acc = {test_acc:.4f}  |  AUC = {auc:.4f}")
    preds = (probs >= 0.5).astype(int)
    print(classification_report(labels, preds, target_names=['Not Recommended', 'Recommended']))

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
for name, res in test_results.items():
    fpr, tpr, _ = roc_curve(res['labels'], res['probs'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Sentiment Analysis')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, test_results.items()):
    preds = (res['probs'] >= 0.5).astype(int)
    cm = confusion_matrix(res['labels'], preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Rec', 'Rec'], yticklabels=['Not Rec', 'Rec'])
    ax.set_title(f'{name}\nAcc={res["acc"]:.3f}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---
## 7. Inference on New Reviews

In [ ]:
def predict_sentiment(text, model, vocab, max_len=MAX_LEN):
    """Predict recommendation probability for a single review."""
    model.eval()
    cleaned = clean_text(text)
    encoded = vocab.encode(cleaned, max_len)
    tensor = torch.tensor([encoded], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logit = model(tensor)
        prob = torch.sigmoid(logit).item()
    label = 'Recommended' if prob >= 0.5 else 'Not Recommended'
    return label, prob

# Load best LSTM model
best_model = LSTMClassifier(
    vocab_size=len(vocab), embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, dropout=DROPOUT
).to(DEVICE)
best_model.load_state_dict(best_models['LSTM'])

sample_reviews = [
    "Absolutely love this dress! The fabric is soft and the fit is perfect. Will buy again!",
    "Terrible quality. The stitching came apart after one wash. Very disappointed.",
    "It's okay. Nothing special but not bad either. Average product for the price.",
    "Beautiful design and very comfortable. Runs a little small so size up.",
    "Worst purchase ever. Material is cheap and it looks nothing like the photos."
]

print(f"{'Review':<75} {'Prediction':<20} {'Prob':>6}")
print('=' * 105)
for review in sample_reviews:
    label, prob = predict_sentiment(review, best_model, vocab)
    short = review[:72] + '...' if len(review) > 72 else review
    print(f"{short:<75} {label:<20} {prob:.3f}")

---
## Summary

| Model | Architecture | Key Strengths |
|-------|-------------|---------------|
| Vanilla RNN | Bidirectional, 2-layer | Fast, but limited on long reviews |
| LSTM | Bidirectional, 2-layer | Best at capturing long-range sentiment cues |
| GRU | Bidirectional, 2-layer | Good balance of speed and accuracy |

### Key Takeaways
- LSTM and GRU significantly outperform vanilla RNN on text classification
- Bidirectional processing helps capture context from both ends of reviews
- The dataset is imbalanced (~82% recommended) — consider weighted loss or oversampling for production use
- Gradient clipping (`max_norm=5.0`) stabilizes training for all RNN variants